In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List, Any
from sklearn.metrics import precision_score#, recall_score, f1_score, roc_auc_score

from counterfactual_fraud_model import (
    OffPolicyEvaluationPipeline,
    OffPolicyEvaluationConfig,
    DataGeneratorConfig,
    LoggingPolicyConfig,
    CounterfactualEstimatorConfig,
    PipelineConfig
)

In [2]:
config = OffPolicyEvaluationConfig(
    data_generator=DataGeneratorConfig(
        # HACK: erase later
        sample_size=10_000,
        random_state=667
    ),
    logging_policy=LoggingPolicyConfig(
        cutoff=0.1,
        exploration_rate=0.05,  # Default, will be overridden in loop
        random_state=667
    ),
    counterfactual_estimator=CounterfactualEstimatorConfig(random_state=42),
    pipeline=PipelineConfig(include_data=False)
)

# Initialize pipeline with configuration
pipeline = OffPolicyEvaluationPipeline(config)


In [3]:
results = pipeline.run_pipeline(cutoff=0.1, exploration_rate=0.01, include_data=True)

In [4]:
investigate = results['data']
investigate.head()

,model_scores,is_fraud,propensity_score,model_action,policy_action
0,3.136570e-04,0,1.0,allow,allow
1,6.579500e-06,0,1.0,allow,allow
2,2.782213e-09,0,1.0,allow,allow
3,1.532518e-01,0,0.0,block,block
4,3.704167e-02,0,1.0,allow,allow


In [5]:
investigate.groupby(['model_action', 'policy_action']).size()

model_action  policy_action
allow         allow            8671
block         allow              10
              block            1319
dtype: int64

In [6]:
y_true = investigate['is_fraud']
y_pred = (investigate['model_action'] == 'block').astype(int)

precision_score(y_true, y_pred)

0.12189616252821671

In [7]:
filtered_data = investigate[investigate['policy_action'] == 'allow']

y_true = filtered_data['is_fraud']
y_pred = (filtered_data['model_action'] == 'block').astype(int)
weights = 1 / filtered_data['propensity_score']

precision_score(y_true, y_pred)

0.0

In [8]:
filtered_data[filtered_data['propensity_score'] != 1]

,model_scores,is_fraud,propensity_score,model_action,policy_action
4700,0.110508,0,0.01,block,allow
5527,0.171498,0,0.01,block,allow
5604,0.357389,0,0.01,block,allow
6829,0.110018,0,0.01,block,allow
6882,0.157852,0,0.01,block,allow
7684,0.664418,0,0.01,block,allow
7975,0.355591,0,0.01,block,allow
8752,0.186285,0,0.01,block,allow
8951,0.192204,0,0.01,block,allow
9519,0.887872,0,0.01,block,allow


In [9]:
results

{'statistics': {'total_transactions': 10000,
  'allowed_transactions': np.int64(8681),
  'blocked_transactions': np.int64(1319),
  'allow_rate': np.float64(0.8681),
  'block_rate': np.float64(0.1319),
  'fraud_rate_overall': np.float64(0.0554),
  'fraud_rate_allowed': np.float64(0.045156088008293974)},
 'ope_metrics': {'precision': {'mean': 0.0,
   'p025': 0.0,
   'p975': 0.0,
   'n_bootstrap': 5000},
  'recall': {'mean': 0.0, 'p025': 0.0, 'p975': 0.0, 'n_bootstrap': 5000},
  'fraud_rate': {'mean': 0.045187798546190275,
   'p025': 0.040895961846710197,
   'p975': 0.049609616535913316,
   'n_bootstrap': 5000},
  'average_precision': {'mean': 0.03515945088306955,
   'p025': 0.029267228499424133,
   'p975': 0.04190569503542922,
   'n_bootstrap': 5000}},
 'parameters': {'data_generator': {'alpha': 0.1,
   'beta_param': 2.0,
   'mean': -0.5,
   'sd': 0.5,
   'sample_size': 10000,
   'random_state': 667},
  'logging_policy': {'cutoff': 0.1,
   'exploration_rate': 0.01,
   'propensity_type': 

In [1]:
from counterfactual_fraud_model.config import (
    SyntheticRetrainingConfig,
    SyntheticOffPolicyEvaluationConfig,
    SyntheticDataConfig,
    ModelConfig,
    LoggingPolicyConfig,
    CounterfactualEstimatorConfig,
    RetrainingConfig,
    RetrainingModelConfig,
    PipelineConfig,
    ModelType,
    RetrainingStrategy
)

# Import the pipeline
from counterfactual_fraud_model.pipelines import SyntheticRetrainingPipeline

In [2]:
config = SyntheticRetrainingConfig(
    base_config=SyntheticOffPolicyEvaluationConfig(
        synthetic_data=SyntheticDataConfig(
            n_samples=20_000,  # Small for quick testing
            n_features=10,
            n_informative=6,
            n_redundant=2,  # Ensure sum doesn't exceed n_features
            n_repeated=0,   # Keep it simple
            random_state=42
        ),
        model=ModelConfig(
            model_type=ModelType.LIGHTGBM,
            random_state=42
        ),
        logging_policy=LoggingPolicyConfig(
            cutoff=0.05,
            exploration_rate=0.1,
            random_state=42
        ),
        counterfactual_estimator=CounterfactualEstimatorConfig(
            n_bootstrap=500,  # Reduced for speed
            random_state=42
        )
    ),
    retraining=RetrainingConfig(
        retrain_test_size=0.5,
        retrain_model=RetrainingModelConfig(
            base_model=ModelConfig(
                model_type=ModelType.LIGHTGBM,
                random_state=42
            ),
            strategy=RetrainingStrategy.FILTERING,
            classification_threshold=0.1
        )
    )
)

pipeline = SyntheticRetrainingPipeline(config)

In [3]:
pipeline.generate_logging_policy_data(
    logging_policy_cutoff=0.05,
    logging_policy_exploration_rate=0.1
)

# Test getter methods
original_results = pipeline.get_original_results()
train_data = pipeline.get_train_policy_data()
test_data = pipeline.get_test_policy_data()

In [4]:
from counterfactual_fraud_model.generators import SyntheticDataGenerator, LoggingPolicyGenerator, create_preprocessor


data_preprocessor = create_preprocessor(
    RetrainingStrategy.WEIGHTING,
    {}
)

# Step 1: Use preprocessor to prepare training data
X_train, y_train, sample_weights = data_preprocessor.prepare_training_data(train_data)

In [5]:
sum(sample_weights != 1)

11

In [8]:
train_data

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,is_fraud,model_scores,propensity_score,model_action,policy_action
5177,-5.297321,2.787527,-1.656700,-2.284550,3.891467,1.446909,-2.433452,1.137938,0.743600,1.129192,0,0.001740,1.0,allow,allow
5330,-0.371050,1.410222,-0.588224,2.713504,1.716045,0.744018,0.233816,-2.341575,0.041916,-3.109706,0,0.002948,1.0,allow,allow
5820,-1.679765,2.331224,2.448036,-0.409720,4.396128,2.823497,0.808184,0.452178,-0.109340,-2.138692,0,0.002274,1.0,allow,allow
3014,0.010515,0.834123,0.004742,-3.857269,-4.278969,-1.400142,-0.690783,0.594423,1.042753,-2.522367,0,0.001888,1.0,allow,allow
3266,-1.999530,0.052362,-0.033023,1.025358,1.816324,-0.860034,-1.657339,-1.622634,-0.604719,-0.079382,0,0.005487,1.0,allow,allow
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2537,-3.035558,1.311298,0.850780,1.218269,2.547604,0.263885,-2.856924,-1.419497,-1.040696,-0.557303,0,0.001025,1.0,allow,allow
8541,1.065731,-0.924578,0.269048,-0.752724,-2.870684,-2.608986,1.686406,-1.270637,0.109697,-1.028902,0,0.002617,1.0,allow,allow
3725,-1.831865,1.561102,-0.224757,-1.150373,2.926595,2.248030,-0.856302,1.439766,-2.269839,0.004194,1,0.002457,1.0,allow,allow
9716,-3.563766,2.620215,-0.681686,0.281127,0.570349,0.356694,-2.389570,-0.647972,0.881382,0.108049,0,0.002055,1.0,allow,allow


In [22]:
test_data[test_data.propensity_score < 1]

,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,is_fraud,model_scores,propensity_score,model_action,policy_action
3786,0.472374,2.074866,-1.447201,-1.223741,0.578438,2.367752,2.714489,1.472535,-0.627219,-1.555317,0,0.693858,0.0,block,block
999,0.106562,-1.649131,-0.849473,0.459230,1.837220,-0.670258,-0.473739,-0.573561,-0.203256,0.428505,0,0.267592,0.0,block,block
2444,1.250176,-4.706892,0.313976,-1.996018,1.569492,-3.278825,-0.300039,-0.822838,-0.797100,-0.157855,0,0.057162,0.0,block,block
3038,-0.751354,-1.591036,-0.693375,0.715944,0.952216,-2.551448,-0.158483,-2.219619,-2.054146,0.012143,0,0.112462,0.0,block,block
4145,-1.066958,3.386156,0.250217,-3.713976,-0.443352,3.513078,1.349022,4.212592,-2.302969,0.937749,1,0.878319,0.0,block,block
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3298,-2.545386,-1.954317,0.079419,0.342899,3.644058,-4.836283,2.092771,-4.691186,-0.731941,-1.260720,1,0.339698,0.0,block,block
61,-0.626293,2.359519,0.965211,3.681722,0.003415,0.144144,2.941277,-2.543297,0.645091,-0.855478,1,0.579834,0.0,block,block
618,-0.108781,0.606774,-0.887500,-0.577799,0.858015,1.567046,-0.391637,1.305505,1.644297,0.190077,0,0.071377,0.0,block,block
9720,-0.919557,0.463526,0.594999,-1.020496,1.575391,0.674381,0.895497,0.926267,1.430035,0.951142,1,0.593175,0.0,block,block


In [ ]:
results = pipeline.run_retrain_pipeline(retraining_config=RetrainingConfig(
            retrain_test_size=0.5,
            retrain_model=RetrainingModelConfig(
                base_model=ModelConfig(
                    model_type=ModelType.LIGHTGBM,
                    random_state=42
                ),
                strategy=RetrainingStrategy.FILTERING,
                classification_threshold=0.1
            )
        ))

Retraining model on training data with config: retrain_test_size=0.3 retrain_model=RetrainingModelConfig(base_model=ModelConfig(model_type=<ModelType.LIGHTGBM: 'lightgbm'>, model_params={}, random_state=42), strategy=<RetrainingStrategy.FILTERING: 'filtering'>, strategy_params={}, classification_threshold=0.1)
Strategy: RetrainingStrategy.FILTERING


In [6]:
results#.keys()

{'original_results': {'model_performance': {'precision': 0.34615384615384615,
   'recall': 0.18,
   'f1': 0.23684210526315788,
   'roc_auc': 0.8143346938775511,
   'average_precision': 0.1940385958239888},
  'dataset_info': {'n_samples': 5000,
   'n_features': 10,
   'n_informative': 6,
   'fraud_rate': np.float64(0.0202),
   'n_fraud': np.int64(101),
   'n_legitimate': np.int64(4899),
   'class_balance': [0.985, 0.015]}},
 'retrained_model_performance': {'precision': 0.3333333333333333,
  'recall': 0.06666666666666667,
  'f1': 0.1111111111111111,
  'roc_auc': 0.6464399092970521,
  'average_precision': 0.1339143511077216},
 'ope_metrics': {'precision': {'mean': 0.30816746031746034,
   'p025': 0.0,
   'p975': 1.0,
   'n_bootstrap': 500},
  'recall': {'mean': 0.06987586853438112,
   'p025': 0.0,
   'p975': 0.24086538461538418,
   'n_bootstrap': 500},
  'fraud_rate': {'mean': 0.017170144374868128,
   'p025': 0.008289004551343047,
   'p975': 0.02774365302997461,
   'n_bootstrap': 500},
  '

In [9]:
pipeline.get_config()

SyntheticRetrainingConfig(base_config=SyntheticOffPolicyEvaluationConfig(synthetic_data=SyntheticDataConfig(n_samples=5000, n_features=10, n_informative=6, n_redundant=2, n_repeated=0, n_clusters_per_class=2, weights=[0.985, 0.015], flip_y=0.01, class_sep=1.0, test_size=0.5, random_state=42), model=ModelConfig(model_type=<ModelType.LIGHTGBM: 'lightgbm'>, model_params={}, random_state=42), logging_policy=LoggingPolicyConfig(cutoff=0.05, exploration_rate=0.1, propensity_type=<PropensityType.UNIFORM: 'uniform'>, random_state=42), counterfactual_estimator=CounterfactualEstimatorConfig(n_bootstrap=500, random_state=42), pipeline=PipelineConfig(include_data=True)), retraining=RetrainingConfig(retrain_test_size=0.3, retrain_model=RetrainingModelConfig(base_model=ModelConfig(model_type=<ModelType.LIGHTGBM: 'lightgbm'>, model_params={}, random_state=42), strategy=<RetrainingStrategy.FILTERING: 'filtering'>, strategy_params={}, classification_threshold=0.1)))

In [7]:
import numpy as np
from typing import List, Dict, Any, Tuple
import pandas as pd
from counterfactual_fraud_model.config import (
    SyntheticRetrainingConfig,
    SyntheticOffPolicyEvaluationConfig,
    SyntheticDataConfig,
    ModelConfig,
    LoggingPolicyConfig,
    CounterfactualEstimatorConfig,
    RetrainingConfig,
    RetrainingModelConfig,
    ModelType,
    RetrainingStrategy
)

from counterfactual_fraud_model.pipelines import SyntheticRetrainingPipeline


def run_retraining_simulations(
    exploration_rates: np.ndarray,
    strategies: List[RetrainingStrategy],
    cutoff: float = 0.05,
    sample_size: int = 300_000,
    classification_threshold: float = 0.1,
    random_state: int = 42
) -> Tuple[List[Dict[str, Any]], pd.DataFrame]:
    """
    Run multiple SyntheticRetrainingPipeline simulations with different exploration rates and strategies.
    
    Args:
        exploration_rates: Array of exploration rate values to test
        strategies: List of retraining strategies to compare
        cutoff: Fixed cutoff value for all simulations
        sample_size: Number of samples in synthetic dataset
        classification_threshold: Threshold for binary classification
        random_state: Random seed for reproducibility
        
    Returns:
        Tuple of (list of simulation results, reference data for plots)
    """
    # Calculate total number of simulations
    total_sims = len(exploration_rates) * len(strategies)
    print(f"Running {total_sims} simulations...")
    print(f"  - Exploration rates: {len(exploration_rates)} values")
    print(f"  - Retraining strategies: {len(strategies)} strategies")
    
    results = []
    reference_data = None  # Will store base data for plotting

    
    # Run simulations for each combination
    for exploration_rate in exploration_rates:
        print(f"Running simulations for exploration_rate={exploration_rate:.3f}")

        # Base config for most simulations
        config = SyntheticRetrainingConfig(
            base_config=SyntheticOffPolicyEvaluationConfig(
                synthetic_data=SyntheticDataConfig(
                    n_samples=sample_size,
                    n_features=15,
                    n_informative=10,
                    n_redundant=3,
                    n_repeated=0,
                    random_state=random_state  # Same seed for fair comparison
                ),
                model=ModelConfig(
                    model_type=ModelType.LIGHTGBM,
                    random_state=random_state
                ),
                logging_policy=LoggingPolicyConfig(
                    cutoff=cutoff,
                    exploration_rate=exploration_rate,
                    random_state=random_state
                ),
                counterfactual_estimator=CounterfactualEstimatorConfig(
                    n_bootstrap=5000,  # Reasonable for analysis
                    random_state=random_state
                )
            )
        )
        
        # Initialize and run pipeline
        pipeline = SyntheticRetrainingPipeline(config)

        pipeline.generate_logging_policy_data()

        for strategy in strategies:
            print(f"Strategy={strategy.value} simulation")

            retraining_config = RetrainingConfig(
                retrain_test_size=0.5,
                retrain_model=RetrainingModelConfig(
                    base_model=ModelConfig(
                        model_type=ModelType.LIGHTGBM,
                        random_state=random_state
                    ),
                    strategy=strategy, # it doesn't matter which strategy we use, it'll change dinamically through the simulation
                    classification_threshold=classification_threshold
                )
            )
            
            result = pipeline.run_retrain_pipeline(retraining_config=retraining_config)  # Removed include_data parameter
        
            # Add metadata for easy tracking
            result['simulation_metadata'] = {
                'exploration_rate': exploration_rate,
                'strategy': strategy.value,
                'result': result
            }
            
            results.append(result)
            
            # Store reference data from first simulation for plotting
            if reference_data is None:
                # Get the base data for plotting (same across all simulations with same random_state)
                reference_data = pipeline.get_test_policy_data()
    
    return results, reference_data

In [8]:
results, reference_data = run_retraining_simulations([0.05], [RetrainingStrategy.FILTERING, RetrainingStrategy.WEIGHTING])

Running 2 simulations...
  - Exploration rates: 1 values
  - Retraining strategies: 2 strategies
Running simulations for exploration_rate=0.050
Strategy=filtering simulation
Strategy=weighting simulation


In [9]:
[result['retrained_model_performance'] for result in results]

[{'precision': 0.5764705882352941,
  'recall': 0.03281982585398526,
  'f1': 0.062103929024081114,
  'roc_auc': 0.7030748678828251,
  'average_precision': 0.10933421624328477},
 {'precision': 0.6935779816513762,
  'recall': 0.2531815137307435,
  'f1': 0.37095191364082436,
  'roc_auc': 0.8234857703315178,
  'average_precision': 0.39141010721257363}]

In [10]:
[result['ope_metrics']['average_precision'] for result in results]

[{'mean': 0.15559791564435246,
  'p025': 0.08131200637063075,
  'p975': 0.2367600475999915,
  'n_bootstrap': 5000},
 {'mean': 0.47916467868314383,
  'p025': 0.3742750968843306,
  'p975': 0.573967341037328,
  'n_bootstrap': 5000}]